In [16]:
import os
import sys
import pandas as pd
import numpy as np

In [17]:
def evaluate_model_to_actual(actual_file, model_file, start_date, end_date):
    """
    Function to compare actual data against a model/baseline over a specified date range.
    
    Parameters:
    actual_file (str): The path to the actual data CSV file.
    model_file (str): The path to the model or baseline data CSV file.
    start_date (str or datetime): The start date for the comparison.
    end_date (str or datetime): The end date for the comparison.
    
    Returns:
    float: The evaluation score comparing the model against actual data over the date range.
    """

    # Load actual and model data from CSV
    actual = pd.read_csv(actual_file).set_index(["Client", "Warehouse", "Product"])
    model = pd.read_csv(model_file).set_index(["Client", "Warehouse", "Product"])

    # Convert columns to datetime
    actual.columns = pd.to_datetime(actual.columns)
    model.columns = pd.to_datetime(model.columns)

    # Ensure indices and columns match
    if not actual.index.equals(model.index):
        print("Indices do not match between actual and model data!")
        raise ValueError("Indices (Client, Warehouse, Product) do not match!")

    # Filter data for the specified date range
    actual_filtered = actual.loc[:, (actual.columns >= pd.to_datetime(start_date)) & (actual.columns <= pd.to_datetime(end_date))]
    model_filtered = model.loc[:, (model.columns >= pd.to_datetime(start_date)) & (model.columns <= pd.to_datetime(end_date))]

    # **Restrict model to have only the same columns (dates) as the actual data**
    model_filtered = model_filtered.loc[:, model_filtered.columns.isin(actual_filtered.columns)]

    # Debug: Check the shape of the filtered data
    print(f"Actual filtered shape: {actual_filtered.shape}")
    print(f"Model filtered shape (after restriction): {model_filtered.shape}")

    # Ensure date columns match after filtering
    if not actual_filtered.columns.equals(model_filtered.columns):
        print("Date columns do not match between actual and model data!")
        print("Actual columns:", actual_filtered.columns)
        print("Model columns:", model_filtered.columns)
        raise ValueError("Date ranges between actual and model do not match!")

    # Calculate error metrics
    abs_err = np.nansum(abs(model_filtered.values - actual_filtered.values))
    err = np.nansum((model_filtered.values - actual_filtered.values))
    score = abs_err + abs(err)
    score /= actual_filtered.sum().sum()

    print(f"Evaluation score for {model_file}: {score:.6f}") # Output score as a percentage
    return score



In [20]:
def baseline_current_price_forecast(data_file, cutoff_date, num_future_weeks):
    """
    Function to generate a baseline forecast using the last available price from the cutoff date,
    ensuring that the forecasted prices are integers.
    
    Parameters:
    data_file (str): The file path to the clean data file.
    cutoff_date (str or datetime): The cutoff date for the actual data to start forecasting.
    num_future_weeks (int): Number of future weeks to forecast.
    
    Returns:
    pd.DataFrame: A pivoted DataFrame containing the baseline forecast with 'Client', 'Warehouse', 'Product' as index
                  and future dates as columns, with forecasted prices as integers.
    """
    # Load data 
    clean_df = pd.read_parquet(data_file)
    
    # Ensure 'ds' column is a datetime type
    clean_df['ds'] = pd.to_datetime(clean_df['ds'])
    
    # Ensure 'Client', 'Warehouse', 'Product' are integers
    clean_df['Client'] = clean_df['Client'].astype(int)
    clean_df['Warehouse'] = clean_df['Warehouse'].astype(int)
    clean_df['Product'] = clean_df['Product'].astype(int)
    
    # Sort the data by 'Client', 'Warehouse', 'Product', and 'ds' (date)
    clean_df = clean_df.sort_values(by=['Client', 'Warehouse', 'Product', 'ds'])
    
    # Filter data up to the cutoff date
    cutoff_date = pd.to_datetime(cutoff_date)
    price_on_cutoff = clean_df[clean_df['ds'] == cutoff_date].copy()
    
    # Step 2: Generate the end date dynamically based on the number of weeks to forecast
    future_end_date = cutoff_date + pd.DateOffset(weeks=num_future_weeks)
    
    # Step 3: Generate weekly dates starting from the cutoff date and ending after the given number of weeks
    future_dates = pd.date_range(start=cutoff_date, end=future_end_date, freq='W-MON')
    
    # Example: Generating a DataFrame with forecasted prices for the dynamic future dates
    forecast_df = pd.DataFrame()

    # Assume we have the data to forecast (price on cutoff date)
    for _, row in price_on_cutoff.iterrows():
        client, warehouse, product, price = row['Client'], row['Warehouse'], row['Product'], row['Price']
        
        # Ensure the price is converted to an integer
        price = int(price)  # Convert price to integer
        
        # Create a DataFrame for each combination with future dates and forecasted prices
        future_data = pd.DataFrame({
            'Client': client,
            'Warehouse': warehouse,
            'Product': product,
            'ds': future_dates,  # Use the dynamically generated future dates
            'Forecasted_Price': price  # Use the last available price from the cutoff date as the forecast, converted to int
        })
        
        # Append to the forecast DataFrame
        forecast_df = pd.concat([forecast_df, future_data])

    # Ensure 'Client', 'Warehouse', and 'Product' are converted to integers (just in case)
    forecast_df['Client'] = forecast_df['Client'].astype(int)
    forecast_df['Warehouse'] = forecast_df['Warehouse'].astype(int)
    forecast_df['Product'] = forecast_df['Product'].astype(int)

    # First, convert the 'ds' (date) column into string format, so it becomes the column headers
    forecast_df['ds'] = forecast_df['ds'].dt.strftime('%m/%d/%Y')

    # Pivot the DataFrame to have 'Client', 'Warehouse', 'Product' as the index and the dates as columns
    pivot_forecast = forecast_df.pivot_table(
        index=['Client', 'Warehouse', 'Product'],  # Use these as the index
        columns='ds',  # The 'ds' column becomes the columns (date)
        values='Forecasted_Price'  # Values are the forecasted prices
    )

    # Reset the index so that 'Client', 'Warehouse', 'Product' become normal columns
    pivot_forecast = pivot_forecast.reset_index()

    # Fill missing values (NaN) with a default value (e.g., 0) before converting to integer
    pivot_forecast = pivot_forecast.fillna(0).astype(int)  # Convert all forecast columns to integers

    return pivot_forecast

# ----------------------------------------------------
# Run the baseline function
# ----------------------------------------------------
# Example usage
dir_path = "/home/ubuntu/anup/projects/enterprise_forecasting/"
file_path = os.path.join(dir_path, 'data', "phase_1_clean_data.parquet")  
cutoff_date = "2024-01-01"  # Set the cutoff date
num_future_weeks = 14  # Number of weeks to forecast

# Generate the baseline forecast
baseline_forecast_df = baseline_current_price_forecast(file_path, cutoff_date, num_future_weeks)
baseline_forecast_df.head()

# Save the pivoted forecast to a CSV file if needed
outfile_path = os.path.join(dir_path, 'data', "baseline_current_price.csv")
baseline_forecast_df.to_csv(outfile_path, index=False)

actual_file = os.path.join(dir_path, 'data', "Phase 1 - Sales.csv")
model_file = os.path.join(dir_path, 'data', "baseline_current_price.csv")

start_date = "2024-01-01"         # Specify the start date for comparison
end_date = "2024-04-08"   

# Evaluate the model against actual data
score = evaluate_model_to_actual(actual_file, model_file, start_date, end_date)

Actual filtered shape: (15053, 1)
Model filtered shape (after restriction): (15053, 1)
Evaluation score for /home/ubuntu/anup/projects/enterprise_forecasting/data/baseline_current_price.csv: 6.529687


In [21]:
def baseline_moving_average_forecast(data_file, cutoff_date, num_future_weeks, window=12):
    """
    Function to generate a baseline forecast using a 12-week moving average up to the cutoff date.
    
    Parameters:
    data_file (str): The file path to the clean data file.
    cutoff_date (str or datetime): The cutoff date for the actual data to start forecasting.
    num_future_weeks (int): Number of future weeks to forecast.
    window (int): The window size for the moving average (default is 12 weeks).
    
    Returns:
    pd.DataFrame: A pivoted DataFrame containing the baseline forecast with 'Client', 'Warehouse', 'Product' as index
                  and future dates as columns, with forecasted prices as integers.
    """
    # Load data 
    clean_df = pd.read_parquet(data_file)
    
    # Ensure 'ds' column is a datetime type
    clean_df['ds'] = pd.to_datetime(clean_df['ds'])
    
    # Ensure 'Client', 'Warehouse', 'Product' are integers
    clean_df['Client'] = clean_df['Client'].astype(int)
    clean_df['Warehouse'] = clean_df['Warehouse'].astype(int)
    clean_df['Product'] = clean_df['Product'].astype(int)
    
    # Sort the data by 'Client', 'Warehouse', 'Product', and 'ds' (date)
    clean_df = clean_df.sort_values(by=['Client', 'Warehouse', 'Product', 'ds'])
    
    # Filter data up to the cutoff date (get all data before or on the cutoff date)
    cutoff_date = pd.to_datetime(cutoff_date)
    filtered_df = clean_df[clean_df['ds'] <= cutoff_date].copy()

    # Step 1: Calculate the 12-week moving average for each Client, Warehouse, Product
    forecast_df = pd.DataFrame()

    # Group by 'Client', 'Warehouse', 'Product'
    grouped = filtered_df.groupby(['Client', 'Warehouse', 'Product'])
    
    for (client, warehouse, product), group in grouped:
        group = group.sort_values(by='ds')  # Ensure sorted by date
        # Calculate the moving average for the last 'window' weeks
        group['Moving_Avg'] = group['Price'].rolling(window=window, min_periods=1).mean()

        # Get the last available moving average before or on the cutoff date
        last_moving_avg = group.loc[group['ds'] == cutoff_date, 'Moving_Avg'].values
        if len(last_moving_avg) > 0:
            avg_price = last_moving_avg[0]
        else:
            avg_price = group['Moving_Avg'].iloc[-1]  # Use the last available moving average if none match the cutoff date

        # Step 2: Generate future dates for forecasting
        future_end_date = cutoff_date + pd.DateOffset(weeks=num_future_weeks)
        future_dates = pd.date_range(start=cutoff_date, end=future_end_date, freq='W-MON')

        # Step 3: Create a forecast DataFrame using the moving average as the forecasted price
        future_data = pd.DataFrame({
            'Client': client,
            'Warehouse': warehouse,
            'Product': product,
            'ds': future_dates,  # Future dates
            'Forecasted_Price': avg_price  # The forecasted price is the moving average
        })

        # Append to the forecast DataFrame
        forecast_df = pd.concat([forecast_df, future_data])

    # Ensure 'Client', 'Warehouse', and 'Product' are integers
    forecast_df['Client'] = forecast_df['Client'].astype(int)
    forecast_df['Warehouse'] = forecast_df['Warehouse'].astype(int)
    forecast_df['Product'] = forecast_df['Product'].astype(int)

    # First, convert the 'ds' (date) column into string format, so it becomes the column headers
    forecast_df['ds'] = forecast_df['ds'].dt.strftime('%m/%d/%Y')

    # Pivot the DataFrame to have 'Client', 'Warehouse', 'Product' as the index and the dates as columns
    pivot_forecast = forecast_df.pivot_table(
        index=['Client', 'Warehouse', 'Product'],  # Use these as the index
        columns='ds',  # The 'ds' column becomes the columns (date)
        values='Forecasted_Price'  # Values are the forecasted prices
    )

    # Reset the index so that 'Client', 'Warehouse', 'Product' become normal columns
    pivot_forecast = pivot_forecast.reset_index()

    # Fill missing values (NaN) with a default value (e.g., 0) before converting to integer
    pivot_forecast = pivot_forecast.fillna(0).astype(int)  # Convert all forecast columns to integers

    return pivot_forecast

# ----------------------------------------------------
# Run the baseline function for Moving Average
# ----------------------------------------------------

dir_path = "/home/ubuntu/anup/projects/enterprise_forecasting/"
file_path = os.path.join(dir_path, 'data', "phase_1_clean_data.parquet")  
cutoff_date = "2024-01-01"  # Set the cutoff date
num_future_weeks = 14  # Number of weeks to forecast

# Generate the baseline forecast based on 12-week moving average
baseline_forecast_df = baseline_moving_average_forecast(file_path, cutoff_date, num_future_weeks)
print(baseline_forecast_df.head())

# Save the pivoted forecast to a CSV file if needed
outfile_path = os.path.join(dir_path, 'data', "baseline_moving_average.csv")
baseline_forecast_df.to_csv(outfile_path, index=False)

actual_file = os.path.join(dir_path, 'data', "Phase 1 - Sales.csv")
model_file = os.path.join(dir_path, 'data', "baseline_moving_average.csv")

start_date = "2024-01-01"  
end_date = "2024-04-08"   

# Evaluate the model against actual data
score = evaluate_model_to_actual(actual_file, model_file, start_date, end_date)


ds  Client  Warehouse  Product  01/01/2024  01/08/2024  01/15/2024  \
0        0          1      367          51          51          51   
1        0          1      639         108         108         108   
2        0          1      655          31          31          31   
3        0          1     1149          34          34          34   
4        0          1     1485          29          29          29   

ds  01/22/2024  01/29/2024  02/05/2024  02/12/2024  02/19/2024  02/26/2024  \
0           51          51          51          51          51          51   
1          108         108         108         108         108         108   
2           31          31          31          31          31          31   
3           34          34          34          34          34          34   
4           29          29          29          29          29          29   

ds  03/04/2024  03/11/2024  03/18/2024  03/25/2024  04/01/2024  04/08/2024  
0           51          51       